<a href="https://colab.research.google.com/github/jrhumberto/000-pep/blob/main/etl_siconfi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
if (!require(pacman)) install.packages("pacman")
pacman::p_load("httr", "jsonlite", "stringr", "dplyr", "tidyr", "knitr", "devtools", "ggplot2")


devtools::install_github("tchiluanda/rsiconfi")

library("dplyr")
library("rsiconfi")

Loading required package: pacman

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘pacman’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Skipping install of 'rsiconfi' from a github remote, the SHA1 (cb105807) has not changed since last install.
  Use `force = TRUE` to force installation



In [ ]:
library(rsiconfi)
library(dplyr)

uf_para_ibge <- c(
  "RO" = "11", "AC" = "12", "AM" = "13", "RR" = "14", "PA" = "15",
  "AP" = "16", "TO" = "17", "MA" = "21", "PI" = "22", "CE" = "23",
  "RN" = "24", "PB" = "25", "PE" = "26", "AL" = "27", "SE" = "28",
  "BA" = "29", "MG" = "31", "ES" = "32", "RJ" = "33", "SP" = "35",
  "PR" = "41", "SC" = "42", "RS" = "43", "MS" = "50", "MT" = "51",
  "GO" = "52", "DF" = "53"
)


tabela_rreo <- get_rreo(year = 2022,
                       period = c(6), # Sexto Bimestre
                       annex = "06",      # Anexo 3 - Demonstrativo da Receita Corrente Líquida
                       entity = uf_para_ibge["CE"],
                       report_tp =2)      # 1-RREO, 2-RREO-Simplificado
View(tabela_rreo)

[1] 1


Joining with `by = join_by(esfera)`


[1] "http://apidatalake.tesouro.gov.br/ords/siconfi/tt/rreo?an_exercicio=2022&nr_periodo=6&co_tipo_demonstrativo=RREO+Simplificado&no_anexo=RREO-Anexo%2006&id_ente=23"
[1] 0


<0 x 0 matrix>

## 2ª Fase - Extrair as variaveis que melhor disgnostiquem que um estado está se endividando.
1. **DCL como % da RCL** — mede o estoque da dívida líquida em relação à receita; é o índice oficial da LRF para controle de endividamento
1. **Dívida Consolidada Líquida (DCL)** — valor absoluto da dívida (DC menos disponibilidades)
1. **Despesa com Pessoal como % da RCL** — proxy de rigidez fiscal; estados que comprometem mais renda com pessoal têm menos folga e tendem a se endividar mais

In [13]:
library(rsiconfi)
library(dplyr)
library(purrr)

anos <- 2018:2024

estados <- tibble(
  uf = c("RO","AC","AM","RR","PA","AP","TO",
         "MA","PI","CE","RN","PB","PE","AL","SE","BA",
         "MG","ES","RJ","SP","PR","SC","RS",
         "MS","MT","GO","DF"),
  cod_ibge = c("11","12","13","14","15","16","17",
               "21","22","23","24","25","26","27","28","29",
               "31","32","33","35","41","42","43",
               "50","51","52","53")
)

coletar <- function(...) tryCatch(get_rgf(...), error = function(e) NULL)

# =============================================================================
# INDICADOR 1: DCL em R$ (Divida Consolidada Liquida - valor absoluto)
# Fonte: RGF Anexo 02, coluna VALOR
# =============================================================================

rgf02_raw <- map_dfr(anos, function(ano) {
  map_dfr(estados$cod_ibge, function(ente) {
    coletar(year = ano, period = 3, annex = "02", entity = ente, power = "E")
  })
})

dcl_valor <- rgf02_raw %>%
  filter(grepl("D.VIDA CONSOLIDADA L", rotulo, ignore.case = TRUE),
         grepl("^VALOR$",              coluna,  ignore.case = TRUE)) %>%
  group_by(uf, ano = as.integer(exercicio)) %>%
  summarise(dcl_valor = max(valor, na.rm = TRUE), .groups = "drop")

message("Indicador 1 coletado: ", nrow(dcl_valor), " registros")

# =============================================================================
# INDICADOR 2: DCL como % da RCL
# Fonte: RGF Anexo 02, coluna % DA RCL
# Limite LRF para estados: 200%
# =============================================================================

dcl_pct_rcl <- rgf02_raw %>%
  filter(grepl("D.VIDA CONSOLIDADA L", rotulo,  ignore.case = TRUE),
         grepl("% DA RCL",             coluna,   ignore.case = TRUE)) %>%
  group_by(uf, ano = as.integer(exercicio)) %>%
  summarise(dcl_pct_rcl = max(valor, na.rm = TRUE), .groups = "drop")

message("Indicador 2 coletado: ", nrow(dcl_pct_rcl), " registros")

# =============================================================================
# INDICADOR 3: Despesa com Pessoal como % da RCL
# Fonte: RGF Anexo 01, coluna % DA RCL
# Limite LRF para estados: 60%
# =============================================================================

rgf01_raw <- map_dfr(anos, function(ano) {
  map_dfr(estados$cod_ibge, function(ente) {
    coletar(year = ano, period = 3, annex = "01", entity = ente, power = "E")
  })
})

pessoal_pct_rcl <- rgf01_raw %>%
  filter(grepl("DESPESA COM PESSOAL TOTAL", rotulo, ignore.case = TRUE),
         grepl("% DA RCL",                 coluna,  ignore.case = TRUE)) %>%
  group_by(uf, ano = as.integer(exercicio)) %>%
  summarise(pessoal_pct_rcl = max(valor, na.rm = TRUE), .groups = "drop")

message("Indicador 3 coletado: ", nrow(pessoal_pct_rcl), " registros")

# =============================================================================
# DATASET FINAL
# =============================================================================

dataset_final <- cross_join(select(estados, uf), tibble(ano = anos)) %>%
  left_join(dcl_valor,      by = c("uf", "ano")) %>%
  left_join(dcl_pct_rcl,   by = c("uf", "ano")) %>%
  left_join(pessoal_pct_rcl, by = c("uf", "ano")) %>%
  arrange(uf, ano)

write.csv(dataset_final, "endividamento_estados_2018_2024.csv", row.names = FALSE)
print(dataset_final)

ERROR: [1m[33mError[39m in `filter()`:[22m
[1m[22m[36mℹ[39m In argument: `grepl("D.VIDA CONSOLIDADA L", rotulo, ignore.case =
  TRUE)`.
[1mCaused by error:[22m
[33m![39m object 'rotulo' not found


In [34]:
library(rsiconfi)
library(dplyr)
library(purrr)

anos <- 2018:2024

estados <- tibble(
  uf = c("RO","AC","AM","RR","PA","AP","TO",
         "MA","PI","CE","RN","PB","PE","AL","SE","BA",
         "MG","ES","RJ","SP","PR","SC","RS",
         "MS","MT","GO","DF"),
  cod_ibge = c("11","12","13","14","15","16","17",
               "21","22","23","24","25","26","27","28","29",
               "31","32","33","35","41","42","43",
               "50","51","52","53")
)

coletar <- function(...) tryCatch(get_rgf(...), error = function(e) NULL)

rgf02_raw <- map_dfr(anos, function(ano) {
  map_dfr(estados$cod_ibge, function(ente) {
    coletar(year = ano, periodicity = "Q", period = 3,
            report_tp = 1, annex = "02", entity = ente, co_power = "E")
  })
})

rgf01_raw <- map_dfr(anos, function(ano) {
  map_dfr(estados$cod_ibge, function(ente) {
    coletar(year = ano, periodicity = "Q", period = 3,
            report_tp = 1, annex = "01", entity = ente, co_power = "E")
  })
})

dcl_valor <- rgf02_raw %>%
  filter(grepl("DIVIDA CONSOLIDADA L|DÍVIDA CONSOLIDADA L", conta, ignore.case = TRUE),
         coluna == "Até o 3º Quadrimestre") %>%
  group_by(uf, ano = as.integer(exercicio)) %>%
  summarise(dcl_valor = max(valor, na.rm = TRUE), .groups = "drop")

pessoal_pct_rcl <- rgf01_raw %>%
  filter(grepl("DESPESA TOTAL COM PESSOAL - DTP", conta, ignore.case = TRUE),
         coluna == "% sobre a RCL Ajustada") %>%
  group_by(uf, ano = as.integer(exercicio)) %>%
  summarise(pessoal_pct_rcl = max(valor, na.rm = TRUE), .groups = "drop")

dataset_final <- cross_join(select(estados, uf), tibble(ano = anos)) %>%
  left_join(dcl_valor,       by = c("uf", "ano")) %>%
  left_join(pessoal_pct_rcl, by = c("uf", "ano")) %>%
  arrange(uf, ano)

write.csv(dataset_final, "endividamento_estados_2018_2024.csv", row.names = FALSE)
print(dataset_final)

Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(esfera)`
Joining with `by = join_by(e

# A tibble: 189 × 4
   uf      ano   dcl_valor pessoal_pct_rcl
   <chr> <int>       <dbl>           <dbl>
 1 AC     2018 3565447362.            48.0
 2 AC     2019 3116891715.            53.7
 3 AC     2020 3337029842.            52.7
 4 AC     2021 2847799048.            51.4
 5 AC     2022 2505321600.            46.4
 6 AC     2023 2051523708.            48.4
 7 AC     2024 1978574844.            46.8
 8 AL     2018 6816346131.            48.7
 9 AL     2019 6404121919.            44.7
10 AL     2020 5813489668.            39.8
# ℹ 179 more rows
